# Galaxy10 DECaLS training with PyTorch

This notebook compares a deliberately small set of CNN configurations on the validation split. The test split is evaluated exactly once after selection. No metric in this unexecuted notebook is a portfolio result.

In [ ]:
from copy import deepcopy
import gc
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch import nn

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from galaxy_classification.galaxy10 import (
    GALAXY10_CLASS_NAMES, GALAXY10_FILENAME, GALAXY10_SHAPE,
    Galaxy10CNN, Galaxy10TrainingConfig, class_weights, make_loaders,
    run_epoch, seed_everything, validate_manifest,
)

ARRAY_PATH = ROOT / 'data' / 'processed' / 'galaxy10-images.npy'
MANIFEST_PATH = ROOT / 'data' / 'image-manifests' / 'galaxy10-decals.csv'
manifest = pd.read_csv(MANIFEST_PATH)
validate_manifest(manifest, GALAXY10_SHAPE[0])
if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA is unavailable. Install requirements-gpu.txt and restart the kernel.'
    )
DEVICE = torch.device('cuda')
print('PyTorch:', torch.__version__)
print('CUDA runtime:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))
DEVICE

## Bounded training-only search

Learning rate, dropout, batch size, and weight decay vary across 24 bounded candidates. The CNN uses three RGB input channels and base_channels=48, producing feature widths 48 -> 96 -> 192 -> 384. The same morphology-preserving augmentation is applied to every training candidate. Class weights are calculated from the training split. Selection uses validation macro-F1 with six-epoch early stopping; the test loader is not touched here.

In [ ]:
CANDIDATES = [
    Galaxy10TrainingConfig(
        learning_rate=learning_rate, base_channels=48, dropout=dropout,
        batch_size=batch_size, weight_decay=weight_decay, epochs=10
    )
    for learning_rate in (1e-3, 3e-4)
    for dropout in (0.3, 0.5, 0.7)
    for batch_size in (64, 128)
    for weight_decay in (0.0, 1e-4)
]
EARLY_STOPPING_PATIENCE = 6
# One worker overlaps loading with GPU work without overcommitting this Windows notebook.
NUMBER_OF_WORKERS = 1
weights = class_weights(manifest).to(DEVICE)
loss_function = nn.CrossEntropyLoss(weight=weights)

In [ ]:
def fit_candidate(config, candidate_number, total_candidates):
    print(f'\nCandidate {candidate_number}/{total_candidates}: learning_rate={config.learning_rate:g}, dropout={config.dropout:.1f}, batch_size={config.batch_size}, weight_decay={config.weight_decay:g}', flush=True)
    seed_everything(config.random_state)
    loaders = make_loaders(
        ARRAY_PATH, manifest, config, number_of_workers=NUMBER_OF_WORKERS
    )
    model = Galaxy10CNN(config.base_channels, config.dropout).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    best_state = None
    best_macro_f1 = -1.0
    stale_epochs = 0
    history = []
    for epoch in range(config.epochs):
        train_metrics = run_epoch(
            model, loaders['train'], loss_function, DEVICE, optimizer
        )
        validation_metrics = run_epoch(
            model, loaders['validation'], loss_function, DEVICE
        )
        history.append({
            'epoch': epoch + 1,
            **{f'train_{key}': value for key, value in train_metrics.items()},
            **{f'validation_{key}': value for key, value in validation_metrics.items()},
        })
        if validation_metrics['macro_f1'] > best_macro_f1:
            best_macro_f1 = validation_metrics['macro_f1']
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }
            stale_epochs = 0
        else:
            stale_epochs += 1
        print(f'  epoch {epoch + 1:02d}/{config.epochs} | train_loss={train_metrics["loss"]:.4f} train_acc={train_metrics["accuracy"]:.4f} | val_loss={validation_metrics["loss"]:.4f} val_acc={validation_metrics["accuracy"]:.4f} val_balanced_acc={validation_metrics["balanced_accuracy"]:.4f} val_macro_f1={validation_metrics["macro_f1"]:.4f} best={best_macro_f1:.4f}', flush=True)
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print('  early stopping: validation metric did not improve', flush=True)
            break
    del model, loaders
    gc.collect()
    torch.cuda.empty_cache()
    print(f'  candidate complete: best validation macro-F1={best_macro_f1:.4f}', flush=True)
    return best_state, pd.DataFrame(history), best_macro_f1

In [ ]:
candidate_results = []
for candidate_number, config in enumerate(CANDIDATES, start=1):
    state, history, best_macro_f1 = fit_candidate(config, candidate_number, len(CANDIDATES))
    candidate_results.append({
        'config': config, 'state': state,
        'history': history, 'validation_macro_f1': best_macro_f1,
    })
print('\nGrid search complete. Validation ranking:', flush=True)
selection_table = pd.DataFrame([
    {
        'learning_rate': result['config'].learning_rate,
        'base_channels': result['config'].base_channels,
        'dropout': result['config'].dropout,
        'batch_size': result['config'].batch_size,
        'weight_decay': result['config'].weight_decay,
        'validation_macro_f1': result['validation_macro_f1'],
    }
    for result in candidate_results
]).sort_values('validation_macro_f1', ascending=False)
selection_table

## Retrain the selected configuration

The grid search selected the configuration using validation macro-F1. Now retrain that configuration for up to 80 epochs, selecting checkpoints with validation accuracy and allowing 25 stale epochs before stopping. The test split remains untouched.

In [ ]:
selected = max(candidate_results, key=lambda item: item['validation_macro_f1'])
final_config = Galaxy10TrainingConfig(
    learning_rate=selected['config'].learning_rate,
    base_channels=selected['config'].base_channels,
    dropout=selected['config'].dropout,
    weight_decay=selected['config'].weight_decay,
    batch_size=selected['config'].batch_size,
    epochs=80,
    random_state=selected['config'].random_state,
)
FINAL_EARLY_STOPPING_PATIENCE = 25
seed_everything(final_config.random_state)
final_loaders = make_loaders(
    ARRAY_PATH, manifest, final_config, number_of_workers=NUMBER_OF_WORKERS
)
final_model = Galaxy10CNN(final_config.base_channels, final_config.dropout).to(DEVICE)
final_optimizer = torch.optim.Adam(
    final_model.parameters(), lr=final_config.learning_rate,
    weight_decay=final_config.weight_decay,
)
best_final_state = None
best_final_accuracy = -1.0
stale_epochs = 0
final_history = []
print(f'\nFinal training: up to {final_config.epochs} epochs, patience={FINAL_EARLY_STOPPING_PATIENCE}, selection=validation_accuracy', flush=True)
for epoch in range(final_config.epochs):
    train_metrics = run_epoch(final_model, final_loaders['train'], loss_function, DEVICE, final_optimizer)
    validation_metrics = run_epoch(final_model, final_loaders['validation'], loss_function, DEVICE)
    final_history.append({'epoch': epoch + 1, **{f'train_{key}': value for key, value in train_metrics.items()}, **{f'validation_{key}': value for key, value in validation_metrics.items()}})
    if validation_metrics['accuracy'] > best_final_accuracy:
        best_final_accuracy = validation_metrics['accuracy']
        best_final_state = {key: value.detach().cpu().clone() for key, value in final_model.state_dict().items()}
        stale_epochs = 0
    else:
        stale_epochs += 1
    print(f'  epoch {epoch + 1:02d}/{final_config.epochs} | train_loss={train_metrics["loss"]:.4f} train_acc={train_metrics["accuracy"]:.4f} | val_loss={validation_metrics["loss"]:.4f} val_acc={validation_metrics["accuracy"]:.4f} val_macro_f1={validation_metrics["macro_f1"]:.4f} best_val_acc={best_final_accuracy:.4f}', flush=True)
    if stale_epochs >= FINAL_EARLY_STOPPING_PATIENCE:
        print('  early stopping: validation accuracy did not improve', flush=True)
        break
final_model.load_state_dict(best_final_state)
final_history = pd.DataFrame(final_history)
print(f'Final training complete: best validation accuracy={best_final_accuracy:.4f}', flush=True)
final_history.tail()

In [ ]:
truth, predictions = [], []
final_model.eval()
with torch.no_grad():
    for inputs, targets in final_loaders['test']:
        logits = final_model(inputs.to(DEVICE, non_blocking=True))
        truth.extend(targets.tolist())
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
report = pd.DataFrame(classification_report(
    truth, predictions, target_names=GALAXY10_CLASS_NAMES,
    output_dict=True, zero_division=0
)).transpose()
display(report)
matrix = confusion_matrix(truth, predictions)
figure, axis = plt.subplots(figsize=(11, 9))
sns.heatmap(matrix, cmap='Blues', ax=axis)
axis.set(xlabel='Predicted class', ylabel='True class', title='Held-out test confusion matrix')
figure.tight_layout()